# Linear elasticity

How does a solid body deform under a force? The unknown $\vec u(x)$ is now a vector: every point receives a displacement.

## Balance, strain, and stress

The balance of forces is

$$
-\operatorname{div}\big(\sigma(\vec u)\big)=f \qquad\text{in }\Omega.
$$

For small deformations, the strain and the isotropic Hooke law are

$$
\varepsilon(\vec u)=\frac12\bigl(\nabla \vec u+\nabla \vec u^T\bigr),
\qquad
\sigma(\vec u)=2\mu\varepsilon(\vec u)+\lambda\operatorname{div}(\vec u)I.
$$

We clamp the left edge, $\vec u= 0$, and pull downward on the right edge. When the Poisson ratio approaches $1/2$, the material becomes nearly incompressible.

In [ ]:
from netgen.occ import OCCGeometry, Rectangle, X
from ngsolve import (
    Mesh, VectorH1, GridFunction, BilinearForm, LinearForm, CF,
    Sym, Grad, Trace, Id, InnerProduct, dx, ds, Norm
)
from ngsolve.webgui import Draw

beam = Rectangle(2, 0.3).Face().Move((0, -0.15, 0))
beam.edges.Min(X).name = "fixed"
beam.edges.Max(X).name = "loaded"
mesh = Mesh(OCCGeometry(beam, dim=2).GenerateMesh(maxh=0.12))
Draw(mesh);

In [ ]:
# Numerical solver
E = 1000.0             # Young's modulus
poisson_ratio = 0.3
mu = E / (2*(1+poisson_ratio))
lam = E*poisson_ratio / ((1+poisson_ratio)*(1-2*poisson_ratio))

def stress(strain):
    return 2*mu*strain + lam*Trace(strain)*Id(2)

space = VectorH1(mesh, order=2, dirichlet="fixed")
u, v = space.TnT()

A = BilinearForm(space)
A += InnerProduct(stress(Sym(Grad(u))), Sym(Grad(v))) * dx
f = LinearForm(space)
f += CF((0, -2)) * v * ds("loaded")
A.Assemble()
f.Assemble()

displacement = GridFunction(space)
displacement.vec.data = A.mat.Inverse(space.FreeDofs(), inverse="sparsecholesky") * f.vec

In [ ]:
Draw(Norm(displacement), mesh, "displacement magnitude", deformation=displacement);

## Observe

- Where is the displacement exactly zero, and why?
- What does happen if you increase the force by a factor 10? Does it look realistic?
- What would change if `E` were increased?

[← Biharmonic plate](03_biharmonic_plate.ipynb) · [Lecture overview](index.ipynb) · [Next: incompressible flow →](05_incompressible_flow.ipynb)